Homework7 Finetune and Evaluation

Fine-Tune

Note: Need to pip install unsloth transformers peft bitsandbytes datasets modelscope

In [ ]:
# 1. Install dependencies
#!pip install unsloth transformers peft bitsandbytes datasets modelscope

# 2. Set the environment variable to redirect downloads to ModelScope
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

import os

from unsloth import FastLanguageModel
from trl import SFTTrainer

from transformers import AutoTokenizer, TrainingArguments
from datasets import load_dataset

# Load the base unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit model in 4-bit mode (dynamic 4-bit quantization)
model_name = "unsloth/Qwen2.5-7B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B", # Note: llama-3.1 is 8B, not 7B
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Load our synthetic Q&A dataset
dataset = load_dataset("json", data_files="synthetic_qa.jsonl", split="train")

# Initialize the trainer for Supervised Fine-Tuning (SFT)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=TrainingArguments(
        output_dir="unsloth/Qwen2.5-7B-qlora-finetuned",
        per_device_train_batch_size=4,   # small batch size for Colab GPU
        gradient_accumulation_steps=4,   # accumulate gradients to simulate larger batch
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        save_strategy="epoch"
    )
)

trainer.train()
model.save_pretrained("Qwen2.5-7B-qlora-finetuned")

Evaluate

In [ ]:
# Define some test questions (ensure these were not exactly in training data)
test_questions = [
    "What is the main hypothesis proposed by the paper on quantum computing?",
    "How did the authors of the deep learning study evaluate their model's performance?",
    "What is the evaluation metric used to measure the performance of MuSeR on the HealthBench dataset?",
    "What is the primary limitation of existing benchmarks in evaluating Full-Duplex Speech Language Models (FD-SLMs)?",
    "How does incorporating knowledge distillation affect the performance of a smaller backbone LLM (e.g., Qwen3-32B) compared to its teacher model?"
    # ... (add total 10 questions)
]

# Load the base and fine-tuned models for inference
base_model = FastLanguageModel.from_pretrained(model_name)  # base 8B model
ft_model = FastLanguageModel.from_pretrained("Qwen2.5-7B-qlora-finetuned")

for q in test_questions:
    prompt_input = f"<|system|>{system_prompt}<|user|>{q}<|assistant|>"
    # Tokenize input and generate output with each model
    input_ids = tokenizer(prompt_input, return_tensors='pt').input_ids.cuda()
    base_output_ids = base_model.generate(input_ids, max_new_tokens=150)
    ft_output_ids  = ft_model.generate(input_ids, max_new_tokens=150)
    # Decode the outputs
    base_answer = tokenizer.decode(base_output_ids[0], skip_special_tokens=True)
    ft_answer   = tokenizer.decode(ft_output_ids[0], skip_special_tokens=True)
    # (Post-process to remove the prompt part if needed)
    base_answer = base_answer.split('<|assistant|>')[-1].strip()
    ft_answer   = ft_answer.split('<|assistant|>')[-1].strip()
    print(f"Q: {q}")
    print(f"Base Model Answer: {base_answer}")
    print(f"Fine-Tuned Model Answer: {ft_answer}")
    print("-" * 60)